In [1]:
# ---------------------------------------------------------
# Cell 1: Import Libraries
# ---------------------------------------------------------
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import ipywidgets as widgets
from IPython.display import display, clear_output
import urllib.request
import zipfile
import os
import warnings
warnings.filterwarnings('ignore') # Keeps our presentation clean from red warning text

print("Step 1 Complete: All libraries imported successfully!")

Step 1 Complete: All libraries imported successfully!


In [7]:
# ---------------------------------------------------------
# Cell 2: Download the MODERN MovieLens Dataset
# ---------------------------------------------------------
def fetch_modern_data():
    url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
    zip_path = "ml-latest-small.zip"
    
    if not os.path.exists("ml-latest-small"):
        print("Downloading modern MovieLens dataset...")
        urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall()
        print("Download complete.")
    else:
        print("Modern dataset found locally.")

fetch_modern_data()

# The modern dataset is cleaner, but removed user demographics for privacy.
ratings = pd.read_csv('ml-latest-small/ratings.csv')
movies = pd.read_csv('ml-latest-small/movies.csv')

# Merge them together
data = pd.merge(ratings, movies, on='movieId')
print(f"Step 2 Complete: Loaded {len(data)} modern movie ratings!")

Download complete.
Step 2 Complete: Loaded 100836 modern movie ratings!


In [8]:
# ---------------------------------------------------------
# Cell 3: Model Training with Genre Extraction
# ---------------------------------------------------------
print("Extracting genres and training the modern model...")

# 1. Calculate user and movie averages
user_avg = data.groupby('userId')['rating'].mean().reset_index().rename(columns={'rating': 'user_avg_rating'})
movie_avg = data.groupby('movieId')['rating'].mean().reset_index().rename(columns={'rating': 'movie_avg_rating'})
data = pd.merge(data, user_avg, on='userId')
data = pd.merge(data, movie_avg, on='movieId')

# 2. Extract Genres! (Turns "Action|Sci-Fi" into separate columns of 1s and 0s)
genre_dummies = data['genres'].str.get_dummies(sep='|')
data = pd.concat([data, genre_dummies], axis=1)

# Get a list of all unique genres for our UI later
all_genres = list(genre_dummies.columns)
if '(no genres listed)' in all_genres: all_genres.remove('(no genres listed)')

# 3. Define our new modern features
features = ['userId', 'movieId', 'user_avg_rating', 'movie_avg_rating'] + all_genres

X = data[features]
y = data['rating']

# 4. Train the model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, learning_rate=0.1, max_depth=5)
model.fit(X_train, y_train)

# 5. Prepare the Movie Catalog for the UI
catalog_cols = ['movieId', 'title', 'genres', 'movie_avg_rating'] + all_genres
movie_catalog = data[catalog_cols].drop_duplicates()

rmse = np.sqrt(mean_squared_error(y_test, model.predict(X_test)))
print(f"Step 3 Complete! Error margin (RMSE): {rmse:.2f} stars.")

Extracting genres and training the modern model...
Step 3 Complete! Error margin (RMSE): 0.79 stars.


In [6]:
# ---------------------------------------------------------
# Cell 4: Interactive Live-Inference Profile Builder
# ---------------------------------------------------------
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- 1. Define the UI Sliders and Buttons ---
age_slider = widgets.IntSlider(value=30, min=10, max=90, description='Age:')
gender_dropdown = widgets.Dropdown(options=[('Female', 0), ('Male', 1)], description='Gender:')
critic_slider = widgets.FloatSlider(value=3.5, min=1.0, max=5.0, step=0.1, description='Critic Level:', 
                                    tooltip='Lower = Hates everything, Higher = Loves everything')
predict_btn = widgets.Button(description="Predict My Top 5!", button_style='primary')
output_area = widgets.Output()

# --- 2. The Prediction Logic ---
def generate_live_predictions(b):
    with output_area:
        clear_output()
        print("Analyzing your unique profile against 1,600+ movies...\n")
        
        # Make a copy of our movie catalog to fill with the user's custom info
        custom_user_data = movie_catalog.copy()
        
        # Apply the user's slider choices to every movie row
        custom_user_data['user_id'] = 999999 # A dummy ID for the live user
        custom_user_data['age'] = age_slider.value
        custom_user_data['gender_num'] = gender_dropdown.value
        custom_user_data['user_avg_rating'] = critic_slider.value
        
        # Ensure the columns match the exact order the model was trained on
        X_live = custom_user_data[features]
        
        # Predict the ratings!
        custom_user_data['predicted_rating'] = model.predict(X_live)
        
        # Sort to find the highest predicted ratings
        top_5 = custom_user_data.sort_values('predicted_rating', ascending=False).head(5)
        
        print("🍿 Based on your profile, you should watch:")
        for index, row in top_5.iterrows():
            stars = round(row['predicted_rating'], 2)
            print(f"⭐️ {stars} / 5.0 --> {row['title']}")

# --- 3. Connect and Display ---
predict_btn.on_click(generate_live_predictions)

ui_layout = widgets.VBox([
    widgets.HTML("<h2>🎬 AI Movie Matchmaker</h2>"),
    widgets.HTML("<p>Adjust the sliders to build a profile. The XGBoost model will instantly predict your favorite movies!</p>"),
    age_slider,
    gender_dropdown,
    critic_slider,
    widgets.HTML("<br>"),
    predict_btn,
    widgets.HTML("<hr>"),
    output_area
])

display(ui_layout)